# Airport Baggage Inspection Queue Simulation

A reproducible next-event simulation for exploring staffing and waiting-time tradeoffs at a secondary inspection station. **Synthetic baseline only** until the airport dataset is obtained and its timestamps are audited.

**Provenance:** AI-generated draft; human review pending.

## 1. Problem and question
How does the number of inspection stations affect average queue wait, the probability of waiting longer than five minutes, and utilisation? This notebook does **not** assume that screening event timestamps are arrivals to the overall checkpoint.

## 2. Model, assumptions and equations
One infinite-capacity FIFO queue feeding $c$ identical parallel stations. Baseline arrivals form a Poisson process with rate $\lambda$, and independent exponential service times have mean $1/\mu$ (M/M/$c$). Events are arrivals and departures. The system is stable in steady state when $\rho=\lambda/(c\mu)<1$. A future empirical-service model samples durations from the audited source data, yielding an M/G/$c$ simulation; M/M/$c$ formulae are **not** an analytic reference for that empirical model.

For the M/M/$c$ benchmark, $a=\lambda/\mu$, $P_0=[\sum_{n=0}^{c-1}a^n/n!+a^c/(c!(1-\rho))]^{-1}$, $P_W=a^c P_0/(c!(1-\rho))$, and $W_q=P_W/(c\mu-\lambda)$.

**Units:** all time quantities in minutes. **Provenance:** AI-generated draft; human review pending.

## 3. Implementation

In [ ]:
from pathlib import Path
import sys, math, statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as student_t
try:
    from src.queue_model import simulate, random_trace
except ModuleNotFoundError:
    from queue_model import simulate, random_trace


In [ ]:
def mmc_wait(lambda_rate, mu_rate, servers):
    if not 0 < lambda_rate < servers * mu_rate:
        raise ValueError("Requires stable M/M/c parameters")
    a = lambda_rate / mu_rate
    rho = a / servers
    p0 = 1 / (sum(a**k/math.factorial(k) for k in range(servers)) + a**servers/(math.factorial(servers)*(1-rho)))
    pw = a**servers * p0/(math.factorial(servers)*(1-rho))
    return pw/(servers*mu_rate-lambda_rate)

def replication(rate, mean_service, servers, n, seed, warmup=1000):
    arrivals, services = random_trace(rate, mean_service, n + warmup, seed)
    result = simulate(arrivals, services, servers=servers)
    # Warm-up is discarded from customer waiting observations only.
    # Area/time statistics require a separate steady-state window implementation.
    return float(np.mean(result.waits[warmup:])), float(np.mean(np.asarray(result.waits[warmup:]) > 5))

def mean_ci(values, alpha=0.05):
    values = np.asarray(values, dtype=float)
    if len(values) < 2: raise ValueError("At least two replications required")
    avg = float(values.mean())
    half = float(student_t.ppf(1-alpha/2, len(values)-1) * values.std(ddof=1)/np.sqrt(len(values)))
    return avg, avg-half, avg+half


## 4. Verification
Visible hand-calculated event trace, deterministic zero-wait case, and analytical M/M/$c$ comparison with a predeclared tolerance. Analytical agreement is statistical, not an exact per-run identity.

In [ ]:
hand = simulate([0, 1, 1.5], [3, 2, 1], servers=1)
assert np.allclose(hand.waits, [0, 2, 3.5])
assert simulate([0, 10, 20], [1, 1, 1], servers=1).mean_wait == 0
assert random_trace(0.8, 1, 10, 7) == random_trace(0.8, 1, 10, 7)
print("Hand trace, zero-wait and seed reproducibility: PASS")

In [ ]:
rate, mean_service, servers = 0.8, 1.0, 2
samples = [replication(rate, mean_service, servers, 12000, seed=100+i)[0] for i in range(30)]
estimate, lower, upper = mean_ci(samples)
analytic = mmc_wait(rate, 1/mean_service, servers)
print(f"Simulated Wq = {estimate:.4f} min; 95% CI [{lower:.4f}, {upper:.4f}]")
print(f"M/M/c analytical Wq = {analytic:.4f} min")
assert abs(estimate - analytic) < 0.035, "M/M/c comparison outside predeclared 0.035-minute tolerance"

## 5. Experiments and results
Independent seeded replications; the confidence intervals quantify Monte Carlo uncertainty for the synthetic baseline, not uncertainty about real-world arrival assumptions.

In [ ]:
records=[]
for c in [1,2,3,4]:
    if rate >= c/mean_service: continue
    rep=[replication(rate, mean_service, c, 5000, 5000+c*100+i) for i in range(30)]
    w=mean_ci([v[0] for v in rep]); tail=mean_ci([v[1] for v in rep])
    records.append(dict(stations=c, mean_wait=w[0], ci_low=w[1], ci_high=w[2], p_wait_over_5=tail[0], utilization_theory=rate*mean_service/c))
results=pd.DataFrame(records)
display(results)
plt.errorbar(results.stations,results.mean_wait,yerr=[results.mean_wait-results.ci_low,results.ci_high-results.mean_wait],fmt='o-',capsize=4)
plt.xlabel('Inspection stations');plt.ylabel('Mean queue wait (minutes)');plt.title('Synthetic M/M/c scenario: 95% t intervals');plt.grid(alpha=.25);plt.show()

## 6. Discussion and limitations
These are synthetic demonstrations, **not airport observations**. The source data must be audited before empirical fitting. Check whether timestamps measure screening starts, finishes, or checkpoint entry; arrival processes and selection into secondary inspection cannot be inferred without appropriate fields. No operational staffing recommendation is warranted from synthetic experiments alone. The current finite-customer warm-up discards early waiting observations but not queue-area counters.

## 7. References
- Gross, Shortle, Thompson and Harris, *Fundamentals of Queueing Theory* (bibliographic details to verify).
- Airport Security Checkpoint Data, 4TU.ResearchData (exact dataset version, DOI, licence and field descriptions to verify).

**References are placeholders pending source verification.**